# 第19周 · 作业：workflow 与 Agent 的评测对比 + Capstone 消融与初稿（预计约5小时）

**研究问题**：在课程的 20 道评测题上，固定步骤的 workflow 与工具调用 Agent 谁更准、谁更省？不用工具的单次调用基线处在什么位置？

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 实验设计：变量分类 + 预先写下的假设 | 0.5 小时 |
| B | 执行：三个系统 × 20 题（真实 Claude 下每题 3 次） | 1 小时 |
| C | 结果表与图 + 分析问题 | 0.5 小时 |
| D | 失败分析：逐条列出错题与类别 | 1 小时 |
| E | 实验记录 | 0.25 小时 |
| F | Capstone：消融实验与报告初稿 | 1.5 小时 |
| G | 代码讲解 | 0.25 小时 |

**两种后端**：没有 API 密钥时，整本 notebook 在 `MockLLM` 上运行——一个按剧本作答、预先注入了若干典型错误的模拟模型（故障的位置和种类都与讲义演示的不同），结果确定、不花钱，你的失败分析会被自动核对。在 Colab 上配置了 Claude 密钥（Secrets）时，同一份代码跑真实模型、每题重复 3 次——这时的数字才是真正的测量，失败分析也无法自动核对，全靠你读轨迹。两种情况都要完成全部问题；用的是哪个后端，要写进结论。

**约定**：可以使用 AI 编程助手，但一定要写 G 部分的代码讲解。**自评**（学习指南里的研究实验自评表）：设计 30% · 执行 30% · 分析与解释 30% · 表达 10%；F 部分对照学习指南里 Capstone 第 19 周的里程碑（消融与初稿）自查。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 本周新增依赖：Anthropic 官方 Python SDK（只有配置了 API 密钥、调用真实 Claude 时才用到；
# 没有密钥时，课程用工具包里的 MockLLM 跑通全部代码）
if importlib.util.find_spec("anthropic") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic>=2.3,<3"], check=True)
print("anthropic SDK 已就绪")

In [ ]:
import json
import time
import anthropic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from zero2researcher import agents, experiments
from zero2researcher.checks import week19 as ck
from zero2researcher.checks import report

MODEL = agents.MODEL
THINKING = {"type": "adaptive", "display": "summarized"}   # thinking 块返回可读的推理摘要
STATUS = {"end_turn": "answered", "refusal": "refused", "max_tokens": "truncated"}
items = agents.load_eval()
item_by_id = {it["id"]: it for it in items}

In [ ]:
llm = agents.make_backend(mock_faults="homework")     # mock 下用作业专用的故障分布
MOCK = isinstance(llm, agents.MockLLM)
N_TRIALS = 1 if MOCK else 3        # mock 是确定性的，重复没有意义；真实模型不能设 temperature，每题跑 3 次
print("后端:", type(llm).__name__, "| 每题重复:", N_TRIALS, "次")

## A · 实验设计（先写，再跑）

实验规格：

- **系统**：单次调用基线 `run_direct`（不给工具）；固定步骤的 `run_workflow`（route → query → wiki_search → expression → calculator → answer）；工具调用 Agent `run_agent`（`max_steps = 8`）；
- **相同的**：模型（`agents.MODEL`）、工具与工具定义（`agents.TOOLS`）、提示词（`agents.AGENT_SYSTEM` / `WORKFLOW_PROMPTS` / `DIRECT_SYSTEM`）、20 道题、判分规则 `agents.grade`；
- **指标**：准确率；每题平均 LLM 调用次数、工具调用次数、输入 / 输出 token；
- **重复**：mock 下 1 次（确定性）；真实 Claude 下每题 3 次，准确率报告"3 次试验准确率的均值 ± 标准差"。

在运行任何代码之前，填写下面两项（**预注册**：跑完之后不许改）：

**A1 · 变量分类**（科学 / 冗余 / 固定，每类至少一项；冗余变量写明你怎样处理它）

- 科学变量：
- 冗余变量：
- 固定变量：

**A2 · 假设**：你预期三个系统的准确率和 token 各自怎样排序？在算术题和查资料题上是否不同？什么结果会推翻你的假设？

- 

## B · 执行

**B1** · 把讲义里写的函数搬过来（可以直接复制）。检查在 MockLLM 上运行，不花钱。

In [ ]:
def new_result():
    return {"answer": None, "llm_calls": 0, "tool_calls": 0, "input_tokens": 0, "output_tokens": 0,
            "trace": [], "stop_reason": None, "status": None}


def run_tool(block):
    # 在这里写你的代码
    raise NotImplementedError


def run_agent(question, llm, max_steps=8):
    # 在这里写你的代码
    raise NotImplementedError


ck.tool_result_block(run_tool)
ck.react_loop(run_agent)

In [ ]:
def ask(llm, step, user_text, res):
    """workflow 的一步（已给出）：以 WORKFLOW_PROMPTS[step] 为 system 调用一次模型，返回回复文字。"""
    response = llm.create(model=MODEL, max_tokens=16000, system=agents.WORKFLOW_PROMPTS[step],
                          messages=[{"role": "user", "content": user_text}])
    res["llm_calls"] += 1
    res["input_tokens"] += response.usage.input_tokens
    res["output_tokens"] += response.usage.output_tokens
    res["stop_reason"] = response.stop_reason
    if response.stop_reason != "end_turn":           # 拒答或被截断：这一步没有可用的输出
        res["status"] = res["status"] or STATUS.get(response.stop_reason, response.stop_reason)
        return ""
    return "".join(b.text for b in response.content if b.type == "text").strip()


def call_tool(name, args, res):
    """由代码决定调用的工具（已给出）：执行、记进 trace，返回输出文字。"""
    try:
        output, is_error = str(agents.TOOL_FUNCTIONS[name](**args)), False
    except Exception as e:
        output, is_error = f"Error: {e}", True
    res["trace"].append({"name": name, "input": args, "output": output, "is_error": is_error})
    res["tool_calls"] += 1
    return output


def run_workflow(question, llm):
    res = new_result()
    notes = []
    # 在这里写你的代码
    raise NotImplementedError
    return res


def run_direct(question, llm):
    """单次调用基线（已给出）。"""
    res = new_result()
    response = llm.create(model=MODEL, max_tokens=16000, system=agents.DIRECT_SYSTEM,
                          messages=[{"role": "user", "content": question}])
    res["llm_calls"] = 1
    res["input_tokens"], res["output_tokens"] = response.usage.input_tokens, response.usage.output_tokens
    res["stop_reason"] = response.stop_reason
    res["status"] = STATUS.get(response.stop_reason, response.stop_reason)
    if response.stop_reason == "end_turn":
        res["answer"] = "".join(b.text for b in response.content if b.type == "text") or None
    return res


ck.workflow(run_workflow)

**B2** · 写 `evaluate(system_fn, items, llm, n_trials=1)`：外层循环试验次数、内层循环题目，每次运行一条记录，字段为 `id`、`type`、`trial`、`answer`、`correct`（**一律用 `agents.grade(item, answer)` 判分**）、`status`、`stop_reason`、`llm_calls`、`tool_calls`、`input_tokens`、`output_tokens`、`trace`，以及 `messages`（只有 Agent 有，其他系统存 `None`）。

真实后端下，一次调用在 SDK 重试两次后仍可能失败（429 限流、5xx、网络错误）。不要让整个评测因此中断：用 `try/except (anthropic.APIConnectionError, anthropic.APIStatusError)` 接住，把它记成一条 `answer=None`、`status="error"`、`error=错误信息` 的记录，继续下一题（`ck.eval_table` 会把这些记录单独计数）；但 400 之类的错误说明代码有问题，应该照样抛出来。

In [ ]:
def evaluate(system_fn, items, llm, n_trials=1):
    records = []
    # 在这里写你的代码
    raise NotImplementedError
    return records


smoke = evaluate(run_direct, items[:2], llm)
print([(r["id"], r["correct"], r["status"], r["llm_calls"]) for r in smoke])

**B3** · 三个系统各跑一遍（真实 Claude 下每题 3 次）：

In [ ]:
SYSTEMS = {"direct": run_direct, "workflow": run_workflow, "agent": run_agent}
SAVE_PATH = None      # 真实后端下建议设成例如 "week19_results.json"：每个系统跑完就存一次，断线后不必全部重跑
results = {}
t0 = time.time()
for name, fn in SYSTEMS.items():
    results[name] = evaluate(fn, items, llm, n_trials=N_TRIALS)
    print(f"{name:9s}: {len(results[name])} 条记录，累计用时 {time.time() - t0:.1f} 秒")
    if SAVE_PATH:                               # messages 里是 SDK 对象，存盘时去掉
        with open(SAVE_PATH, "w", encoding="utf-8") as f:
            json.dump({k: [{c: v for c, v in r.items() if c != "messages"} for r in recs]
                       for k, recs in results.items()}, f, ensure_ascii=False, default=str)
ck.eval_table(results)

## C · 结果表与图

**C1** · 写 `summarize(results)`，返回一个 DataFrame：行 = 系统，列 =

- `accuracy`：全部记录的平均准确率；`accuracy_std`：**各次试验准确率**的标准差（用 `experiments.mean_std`；只有 1 次试验时为 0）；
- `acc_arithmetic`、`acc_lookup`：两类题各自的准确率；
- `mean_llm_calls`、`mean_tool_calls`、`mean_input_tokens`、`mean_output_tokens`：每题平均值。

In [ ]:
def summarize(results):
    # 在这里写你的代码
    raise NotImplementedError


summary = summarize(results)
ck.eval_table(results, summary)
summary.round(3)

**C2** · 画一张两联图：左边是三个系统在两类题上的准确率（分组柱状图；有多次试验时加误差棒），右边是每题平均 token（输入与输出堆叠）。轴标签用英文，**标题写结论**（可以用 f-string 从数字里生成）。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


**C3 · 分析问题**（每题两三句话，引用你表里的数字；写明你用的是 mock 还是真实 Claude）

1. **准确率**：哪个系统最高？差异是否大于试验间的标准差？（mock 下只有 1 次试验——为什么这时谈不上"显著"？）
2. **成本**：Agent 每题的输入 token 是 workflow 的几倍？从消息的结构解释原因（提示：每一步重发了什么）。
3. **分题型**：算术题和查资料题上，两个系统的优劣是否相同？
4. **基线**：不用工具的单次调用在哪类题上输得最多？在查资料题上它和带工具的系统比如何？如果基线已经接近满分，这个评测集还能区分两个系统吗？要想测出"检索真的有用"，你会怎样改评测集？
5. **选择**：*Building Effective Agents* 建议"先找最简单的方案"。根据你的数据，这 20 道题该用哪个系统？什么样的任务会让你改变选择？

（在这里回答）

## D · 失败分析：逐条列出错题与类别

只看准确率，你只知道"错了几道"；读轨迹，才知道"为什么错、该改什么"。

**先看这次运行怎样结束**（`status`）：拒答（`"refused"`）、被截断（`"truncated"`）、用完步数（`"max_steps"`）不属于下面五类里的任何一类，单独标注为"拒答""截断""超步数"。其余的失败，每条只标**一个主因**：

| 类别 | 定义 |
| --- | --- |
| 工具选择错误 | 调用了不该用的工具；workflow 里路由走错了分支 |
| 参数错误 | 工具选对了，但传进去的参数错了（算式列错、检索词查不到所需条目） |
| 提前结束 | 还没拿到所需信息（或没做完所需的计算）就给出答案 |
| 幻觉 / 编造 | 观察结果里已有正确信息，答案却与之矛盾，或凭空捏造 |
| 格式不匹配 | 答案内容其实对了，只是判分器没认出来 |

判断顺序（第一条成立的就是主因）：① 内容对、形式错 → 格式不匹配；② 用错了工具 → 工具选择错误；③ 工具对、参数错 → 参数错误；④ 没拿到所需信息就作答 → 提前结束；⑤ 有信息却答错 → 幻觉 / 编造。

**D1** · 先列出 agent 与 workflow 的全部错题（已给出）：

In [ ]:
fail_rows = []
for system in ("agent", "workflow"):
    for rec in results[system]:
        if not rec["correct"]:
            fail_rows.append({"system": system, "id": rec["id"], "trial": rec["trial"], "status": rec["status"],
                              "gold": item_by_id[rec["id"]]["gold"],
                              "answer (last line)": ((rec["answer"] or "").strip().splitlines() or [""])[-1],
                              "tools used": [t["name"] for t in rec["trace"]]})
print(f"共 {len(fail_rows)} 条失败")
pd.DataFrame(fail_rows)

**D2** · 逐条读轨迹（已给出；Agent 打印完整对话，workflow 打印工具调用记录与最终回答）。**先读，再分类。**

In [ ]:
def show_failure(system, rec, max_lines=12):
    print(f"========== {system} / {rec['id']}（试验 {rec['trial']}）· 标准答案 {item_by_id[rec['id']]['gold']} · "
          f"status={rec['status']!r} · stop_reason={rec['stop_reason']!r}")
    if rec["messages"]:
        agents.print_transcript(rec["messages"])
    else:
        print("[user]", item_by_id[rec["id"]]["question"])
        for t in rec["trace"]:
            print(f"  tool {t['name']}({t['input']}) → {t['output'][:100]}")
        print("[answer]", (rec["answer"] or "").replace("\n", " ⏎ "))
    print()


shown = 0
for system in ("agent", "workflow"):
    for rec in results[system]:
        if not rec["correct"] and shown < 12:
            show_failure(system, rec)
            shown += 1

**D3** · 把你的判断填进 `failure_labels`：键是 `(系统, 题号)`，值是五个类别之一，或非正常结束的"拒答 / 截断 / 超步数"之一（中英文都可）。同一题在多次试验中都失败时，标注最常见的主因。mock 后端下，检查会把你的标注与注入的故障逐条核对；真实 Claude 下无法自动核对，请在 D4 的表里写清证据。

In [ ]:
failure_labels = {}     # 格式示例：failure_labels[("agent", "a00")] = "参数错误"（a00 不是真实题号）
# 在这里写你的代码
raise NotImplementedError
failed = {(row["system"], row["id"]) for row in fail_rows}
report("每道错题都有标注", failed <= set(failure_labels), f"还没有标注：{sorted(failed - set(failure_labels))}")
if MOCK and llm.faults == "homework":
    for (system, qid), label in sorted(failure_labels.items()):
        ck.failure_category(label, f"{system}/{qid}")
else:
    print("真实模型的失败无法自动核对：请在 D4 的表里为每个标注写出轨迹中的证据。")

**D4 · 失败分析表**（每条错题一行）

| 系统 | 题号 | 类别 | 轨迹里的证据（引用一句） | 可检验的改进假设（只改一处） |
| --- | --- | --- | --- | --- |
| | | | | |
| | | | | |

**D5 · 问题**

1. 两个系统各自最多的是哪一类失败？哪些失败方式 workflow 让模型**无法发生**（在什么前提下成立？），哪些是它**新引入**的？非正常结束（拒答、截断、超步数）各出现在哪个系统里，为什么？
2. 从表里挑一个你认为最容易修的失败，写出改进措施与检验方法：改什么（只改一处）、在哪些题上重跑、看什么指标、预期看到什么。
3. （选做，仅限真实 Claude）实施这个改进并重跑 3 次，报告结果。注意：这时你在比较两个提示词，其余一切都要保持不变。mock 的剧本不会因为你改了提示词而改变，所以 mock 下不做这一题。

（在这里回答）

## E · 实验记录

按第 6 周起的实验记录格式填写。"结果"一栏写清用的是哪个后端；真实 Claude 下写均值 ± 标准差（3 次试验）。

## 实验记录 #__ · 日期：____

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量如何处理）：
- 基线：
- 数据与划分方式（评测集来源；判分规则）：
- 随机性与重复次数：
- 结果（数值：均值±标准差；图：链接或插图）：
- 解释：
- 局限：
- 下一步：

## F · Capstone：消融实验与报告初稿

本周的 Capstone 里程碑是**消融实验 + 结果图 + 报告初稿**（学习指南 6.1 节）。报告按 `templates/capstone_report.md` 写：4 页简化会议论文格式（Abstract / Introduction / Related Work / Method / Experiments / Limitations，外加不计页数的 AI 助手使用声明与 Reproducibility）。本周至少完成 Method 与 Experiments 两节，其余写出提纲。

**F1 · 消融表**：从完整方法出发，一次只拿掉（或替换）一个组件，其余设置完全相同（同一组 ≥ 3 个种子、同一划分、同一训练预算）。

| 设置 | 指标（均值 ± 标准差，n = 种子数） | 相对完整方法的变化 | 差异 > 1 个标准差？ |
| --- | --- | --- | --- |
| 完整方法 | | — | — |
| − 组件 A | | | |
| − 组件 B | | | |

**F2 · 结果图**：主结果图与消融图各一张（插入图片或给出 notebook 中的位置）。自查：轴标签与单位、误差棒、图例、结论式标题。

**F3 · 初稿**：报告初稿的链接；以及你**最没把握**的一处（一个结论、一张图、一个设计选择），定稿前请学习伙伴或 AI 审稿人重点看这里（自己重读时也先看这里）。

**F4 · 互查**（按讲义 S4 的互查清单逐项检查——和学习伙伴交换初稿，或隔一天后自己当审稿人重读一遍；把发现的问题和修改计划写在这里）：

- 发现的问题：
- 我的修改计划：

（在这里填写 F1–F4）

## G · 代码讲解（每份作业都写）

用你自己的话写一段（约 150–300 字）：

1. 逐段解释 `run_agent`：为什么先看 `stop_reason`，`status` 怎样区分正常作答、拒答、截断与用完步数；为什么要把 `response.content` **原样**追加（删掉 thinking 块、改写之前的轮次各有什么后果）；为什么同一轮的所有 `tool_result` 应放进同一条 user 消息；`max_steps` 防止了什么；
2. `evaluate` 里的 `correct` 是怎么算的？为什么三个系统必须用同一个判分函数？
3. 注明哪些代码由 AI 助手生成、你对它做了哪些检查或修改；写出一处你一开始没看懂、后来弄懂的地方。

（在这里写）